# CSP Lectures

## CSP Basics

### $\eta_{th}$ as a function of C, $\epsilon$, and $T_{r}$

In [ ]:
from enn554.constants import h,c,kB,σ
from enn554.sun import _set_azimuth_zero_to_north,solar_vector_from_angles, angle_of_incidence
from pysolar import solar
from datetime import datetime, timezone, timedelta
import numpy as np
import matplotlib.pyplot as plt
from enn554.math import tand, cosd, sind, vecnorm, proj
from enn554.paths import data_dir, outputs_dir
dd = data_dir()
od = outputs_dir()

ε,h = 0.94,0
T_amb,T_sky = 300,300-10
η_opt = 0.9
DNI = 1000
η_th = lambda C,Tr,ε: 1 - (σ*ε*(Tr**4-T_sky**4)+h*(Tr-T_amb))/(η_opt*C*DNI)
η_carnot = lambda Tr: (1-T_amb/Tr)
η_ideal = lambda C,Tr,ε: η_opt *η_carnot(Tr) * η_th(C,Tr,ε)



### $C$ and $T_r$

In [ ]:
CRs = [1,2,5,10,20,50,100,200,500,1000,2000,5000,10000]
Trs = np.linspace(T_sky,1500,5000)
norm = plt.Normalize(0,max(np.log10(CRs)))

height = 5.0
fig,ax = plt.subplots(figsize=(16/9*height,height))
for c in CRs:
    η = η_th(c,Trs,ε)
    color = plt.cm.viridis(norm(np.log10(c)))
    ax.plot(Trs[η>=0],η[η>=0],color=color)

ax.set_ylim((0,1))
ax.set_xlim((T_sky,Trs.max()))
ax.set_xlabel('Receiver temperature, $T_r$ (K)')
ax.set_ylabel(r'Receiver efficiency, $\eta_{rec}$')
ax.set_title(f'The effect of Concentration Ratio and Receiver Temperature \n $\\epsilon$={ε:.2f}, $T_{{amb}}$={T_amb:.0f}K, $T_{{sky}}$={T_sky:.0f}K, $\\eta_{{opt}}$={η_opt:.2f}, DNI={DNI:.0f} W/$m^2$',
             fontsize=12)

sm = plt.cm.ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm,ax=ax)
c_ticks = cbar.ax.set_yticks(np.log10(CRs))
cbar.ax.set_yticklabels(CRs)
cbar.ax.set_ylabel("Concentration Ratio")

### Add emissivity effect

In [ ]:
from matplotlib.lines import Line2D
CRs = [1,10,100,1000,10000]
Trs = np.linspace(T_sky,1500,5000)
εs = [0.94,0.5,0.1]
styles = ['-','--',':']
norm = plt.Normalize(0,max(np.log10(CRs)))

height = 5.0
fig,ax = plt.subplots(figsize=(16/9*height,height))
legend_elements = []
for ii,c in enumerate(CRs):
    for jj,e in enumerate(εs):
        η = η_th(c,Trs,e)
        color = plt.cm.viridis(norm(np.log10(c)))
        le = ax.plot(Trs[η>=0],η[η>=0],color=color,linestyle=styles[jj])
        if ii == 1:
            legend_elements.append(Line2D([0],[0],color='black',linestyle=styles[jj],label=f"$\\epsilon$={e:.2f}"))

ax.legend(handles=legend_elements)    

ax.set_ylim((0,1))
ax.set_xlim((T_sky,Trs.max()))
ax.set_xlabel('Receiver temperature, $T_r$ (K)')
ax.set_ylabel(r'Receiver efficiency, $\eta_{rec}$')
title = f'The effect of Concentration Ratio, Emissivity, and Receiver Temperature \n '
title += f'$T_{{amb}}$={T_amb:.0f}K, $T_{{sky}}$={T_sky:.0f}K, $\\eta_{{opt}}$={η_opt:.2f}, DNI={DNI:.0f} W/$m^2$'
ax.set_title(title,fontsize=12)

sm = plt.cm.ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm,ax=ax)
c_ticks = cbar.ax.set_yticks(np.log10(CRs))
cbar.ax.set_yticklabels(CRs)
cbar.ax.set_ylabel("Concentration Ratio")


### Ideal efficiency

In [ ]:
CRs = [1,2,5,10,20,50,100,200,500,1000,2000,5000,10000]
Trs = np.linspace(T_sky,1500,5000)
norm = plt.Normalize(0,max(np.log10(CRs)))

height = 5.0
fig,ax = plt.subplots(figsize=(16/9*height,height))
for c in CRs:
    η = η_ideal(c,Trs,ε)
    color = plt.cm.viridis(norm(np.log10(c)))
    ax.plot(Trs[η>=0],η[η>=0],color=color)

ax.plot(Trs,η_carnot(Trs),label='Carnot',color='black')

ax.set_ylim((0,1))
ax.set_xlim((T_sky,Trs.max()))
ax.set_xlabel('Receiver temperature, $T_r$ (K)')
ax.set_ylabel(r'Ideal efficiency, $\eta_{ideal}$')
ax.set_title(f'Collection + conversion (i.e. ideal) efficiency of the system \n $\\epsilon$={ε:.2f}, $T_{{amb}}$={T_amb:.0f}K, $T_{{sky}}$={T_sky:.0f}K, $\\eta_{{opt}}$={η_opt:.2f}, DNI={DNI:.0f} W/$m^2$',
             fontsize=12)

sm = plt.cm.ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm,ax=ax)
c_ticks = cbar.ax.set_yticks(np.log10(CRs))
cbar.ax.set_yticklabels(CRs)
cbar.ax.set_ylabel("Concentration Ratio")
ax.legend()

## Parabolic trough

### Geometry

In [ ]:
W = 1
ratio = [1.9,0.933,0.603,0.25,0.1442,0.067]
fig,ax = plt.subplots()

for ii,r in enumerate(ratio):
    f = r*W
    ψ_rim = np.arctan2(8*r,16*r**2-1)*180/np.pi
    ψ = np.linspace(-ψ_rim,ψ_rim,1000)
    pp = 2*f/(1+cosd(ψ))
    if ii == 1:
        ax.plot(pp*sind(ψ),-pp*cosd(ψ),color='black',label='parabola')
        ax.plot(0,0,'.',markersize=10,color='blue',label='Focus')
    else:
        ax.plot(pp*sind(ψ),-pp*cosd(ψ),color='black',label=None)
    
    ax.plot([0,pp[-1]*sind(ψ[-1])],[0,-pp[-1]*cosd(ψ[-1])],ls=':',linewidth=0.5,color='green')
    ax.annotate(f"{r:.3g}",(-1.1*W/2,-pp[0]*cosd(ψ[0])),ha='right')
    ax.annotate(rf"{ψ_rim:.0f}$\degree$",(1.1*W/2,-pp[-1]*cosd(ψ[-1])),ha='left',color='green')


ax.legend()
ax.axis('equal')
ax.set_xlim((-W/2,W/2))
ax.axvline(-1.05*W/2,ls=':',linewidth=1.0,color='black')
ax.axvline(1.05*W/2,ls=':',linewidth=1.0,color='green')
ax.axis('off')

y_mid = 0.5*sum(ax.get_ylim())
ax.annotate(r"Rim angle, $\psi_{rim}$",(1.8*W/2,y_mid),ha='center',va='center',rotation=-90,color='green')
ax.annotate(r"Focal length to aperture, $f/W$",(-2.2*W/2,y_mid),ha='center',va='center',rotation=90)


## Central Receiver
### Aiming
Get sun vector first

In [ ]:
# Brisbane, Australia (27.4705 S, 153.0260 E)
lat = -27.47
lon_st = 360-150 # E -> subtract from 360
lon_pysolar = 153.02
lon = 360-153.0260 # E -> subtract from 360
utc_offset = 10 # hours

# local time
s = 0
m = 0
h = 12
day = 1
month = 1
year = 2014

dt = datetime(year,month,day,h,m,s,tzinfo=timezone(timedelta(hours=utc_offset)))
doy = dt.timetuple().tm_yday # day of year
γ_s_pysolar,altitude = solar.get_position(lat,lon_pysolar,dt) 

# convert to beckman convention, south zero, west positive
γ_s = γ_s_pysolar - 180
θ_z = 90-altitude
print(f"Solar zenith: {θ_z:.1f} degrees, Solar azimuth: {γ_s:.1f} (+West of South)")


In [ ]:
from enn554.central_tower import heliostat_normal

In [ ]:
S = solar_vector_from_angles(γ_s,θ_z,convention='sam')
xh,yh,zh = 100,100,1.0
xa,ya,za = 0,0,151
R = np.array([xa-xh,ya-yh,za-zh])
S /= vecnorm(S)
R /= vecnorm(R)

# compute normal
N = S+R
N /= vecnorm(N)
N_rabl = (S+R)/np.sqrt(2+2*np.dot(S,R))

# check consistancy with Power from the Sun
theta = np.arccos(np.dot(N,S))
N_pfs = (S+R)/2/np.cos(theta)

N_func = heliostat_normal([xh,yh,zh],[xa,ya,za],S)

N,N_rabl,N_pfs, N_func

### Blocking / Shading demo with SolTrace
The below requres pysoltrace which requires quite a bit of setup. The below demo was infofmed by tower-demo.py [here](https://github.com/NatLabRockies/SolTrace/blob/develop/app/deploy/api/examples/tower-demo.py)

First, create optical surfaces and sun position

In [ ]:
from pysoltrace import Point, PySolTrace
import plotly.io as pio
pio.renderers.default = "browser"

θ_z = 55
γ_s = -155
rec_h = 20.0
rec_L,rec_W = 3.0,3.0
L,W = 2.0,1.0

helio_positions = [(0.0,1.0,-95.0),(0.0,1.0,-100.0),(-1.5,1.0,-93.0)]
aimpoints = [(0,0,0),(0.0,0,0),(0,0,0)]
abs_pos = Point(0.0,rec_h,0.0)



### Pysoltrace setup

In [ ]:
PT = PySolTrace()
opt_heliostat = PT.add_optic('Reflective surface')
opt_heliostat.front.reflectivity=0.95
opt_heliostat.back.reflectivity=0.0

opt_absorber = PT.add_optic('Absorber')
opt_absorber.front.reflectivity = 0.0
opt_absorber.back.reflectivity = 0.0

S = solar_vector_from_angles(γ_s,θ_z,convention='soltrace')
sun = PT.add_sun() # Z to north, Y to zenith, X to west
sun.position.x, sun.position.y, sun.position.z = S*10000


### Add virtual stage to be certain that SolTrace uses a sane bounding box

In [ ]:
def virtual_aperture(helio_positions, helio_W, helio_L, sun_vec, margin=2.0):
    """
    Returns position, aim, width, height for a virtual aperture element
    perpendicular to sun_vec that covers all heliostats.
    
    sun_vec : unit vector pointing FROM sun TOWARD scene (ray travel direction)
    """
    pos = np.array(helio_positions)
    sun = np.array(sun_vec, dtype=float)
    sun /= np.linalg.norm(sun)

    # Orthonormal basis for the plane perpendicular to sun
    up = np.array([0., 1., 0.])
    if abs(np.dot(sun, up)) > 0.99:
        up = np.array([0., 0., 1.])
    u = np.cross(sun, up);  u /= np.linalg.norm(u)
    v = np.cross(sun, u);   v /= np.linalg.norm(v)

    # Project heliostat positions onto perpendicular plane
    half_diag = 0.5 * np.sqrt(helio_W**2 + helio_L**2)
    proj_u = []
    proj_v = []
    for p in pos:
        pu = np.dot(p, u)
        pv = np.dot(p, v)
        proj_u.extend([pu - half_diag, pu + half_diag])
        proj_v.extend([pv - half_diag, pv + half_diag])

    proj_u = np.array(proj_u)
    proj_v = np.array(proj_v)

    width  = proj_u.max() - proj_u.min() + margin
    height = proj_v.max() - proj_v.min() + margin
    cu = (proj_u.max() + proj_u.min()) / 2
    cv = (proj_v.max() + proj_v.min()) / 2

    # Place aperture upstream of all heliostats along ray direction
    # dot(p, sun) = depth along ray; minimum = closest to sun = most upstream
    t = min(np.dot(p, sun) for p in pos) - 10.0
    centre = cu * u + cv * v + t * sun

    # Aim point: subtract sun so normal points back toward the sun (-sun direction)
    aim = centre - sun

    return centre, aim, width, height

In [ ]:
VIRTUAL_STAGE = 1

centre, aim, ap_W, ap_L = virtual_aperture(
    helio_positions, W, L, -S, margin=2.0
)


st_virtual = PT.add_stage()
st_virtual.is_virtual = True   
opt_virtual = PT.add_optic('virtual')  # dummy, won't be used
opt_virtual.front.reflectivity = 0
opt_virtual.front.transmissivity = 1
opt_virtual.back.reflectivity = 0
opt_virtual.back.transmissivity = 1


ev = st_virtual.add_element()
ev.optic = opt_virtual
ev.position.x, ev.position.y, ev.position.z = centre
ev.aim.x, ev.aim.y, ev.aim.z = aim

ev.surface_flat()
ev.aperture_rectangle(ap_W, ap_L)

### Add heliostats and flate plate receiver

In [ ]:
st = PT.add_stage()
for ii,h in enumerate(helio_positions):
    n = heliostat_normal(h,[abs_pos.x+aimpoints[ii][0],
                            abs_pos.y+aimpoints[ii][1],
                            abs_pos.z+aimpoints[ii][2]],S)
    
    el = st.add_element()
    el.optic = opt_heliostat
    el.position.x = h[0]
    el.position.y = h[1]
    el.position.z = h[2]

    el.aim = Point(el.position.x + 1000*n[0],el.position.y+1000*n[1],el.position.z+1000*n[2]) 
    # el.zrot = PT.util_calc_zrot_azel(list(n))
    el.zrot = 0
    el.surface_flat()
    el.aperture_rectangle(W,L)
    

# sta = PT.add_stage()

# flat absorber element
ela = st.add_element()
ela.position = abs_pos
ela.aim.z = -5.
ela.aim.x = 0.
ela.aim.y = rec_h # rec_h # THIS IS TRICKY!!!!
ela.optic = opt_absorber
ela.surface_flat()
ela.aperture_rectangle(rec_L,rec_W)  

# set simulation parameters
PT.num_ray_hits = int(1e6)
PT.max_rays_traced = int(PT.num_ray_hits*100)
PT.is_sunshape = True 
PT.is_surface_errors = True

### Geometry Visualisation

Visualise the two-heliostat + receiver geometry in 3-D **without** running SolTrace via Plotly. Plotly is difficult to use for me, so this is mostly vibe coded for now. 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
def get_colors(n: int) -> list:
    cmap = plt.get_cmap('tab10')  # or 'tab20', 'Set1', 'hsv', etc.
    return [mcolors.to_hex(cmap(i / max(n - 1, 1))) for i in range(n)]

In [ ]:
import plotly.graph_objects as go

# Helper: 4 corners of a rectangular panel given centre, outward normal, and dims
def _panel_corners(center, normal, width, length):
    N = np.array(normal, dtype=float); N /= np.linalg.norm(N)
    ref = np.array([0., 1., 0.]) if abs(N[1]) < 0.9 else np.array([1., 0., 0.])
    u = np.cross(ref, N); u /= np.linalg.norm(u)
    v = np.cross(N, u);   v /= np.linalg.norm(v)
    c = np.array(center, dtype=float)
    return np.array([c - width/2*u - length/2*v,
                     c + width/2*u - length/2*v,
                     c + width/2*u + length/2*v,
                     c - width/2*u + length/2*v])

# Recompute and store heliostat normals (SolTrace convention: X=West, Y=Up, Z=North)
helio_normals = [
    heliostat_normal(hpos,
                     [abs_pos.x + aimpoints[ii][0],
                      abs_pos.y + aimpoints[ii][1],
                      abs_pos.z + aimpoints[ii][2]], S)
    for ii, hpos in enumerate(helio_positions)
]
rec_pos  = np.array([abs_pos.x, abs_pos.y, abs_pos.z])
rec_norm = np.array([0., 0., -1.])   # receiver faces south (toward heliostats)

HELIO_COLORS = get_colors(len(helio_positions))
# HELIO_COLORS = ['steelblue', 'darkorange']
N_SCALE   = 5.0   # normal vector display length [m]
SUN_SCALE = 8.0   # indicative incoming-ray length [m]

def build_geometry_fig():
    traces  = []
    all_pts = [np.array([[rec_pos[0], 0., rec_pos[2]], rec_pos])]

    for ii, (hpos, N) in enumerate(zip(helio_positions, helio_normals)):
        pts = _panel_corners(hpos, N, W, L)
        all_pts.append(pts)
        hp = np.array(hpos, dtype=float)

        traces.append(go.Mesh3d(
            x=pts[:,0], y=pts[:,1], z=pts[:,2],
            i=[0,0], j=[1,2], k=[2,3],
            color=HELIO_COLORS[ii], opacity=0.85,
            name=f'Heliostat {ii+1}'))

        tip = hp + N * N_SCALE
        traces.append(go.Scatter3d(
            x=[hp[0], tip[0]], y=[hp[1], tip[1]], z=[hp[2], tip[2]],
            mode='lines', line=dict(color=HELIO_COLORS[ii], width=5),
            showlegend=False))
        traces.append(go.Cone(
            x=[tip[0]], y=[tip[1]], z=[tip[2]],
            u=[N[0]], v=[N[1]], w=[N[2]],
            colorscale=[[0, HELIO_COLORS[ii]], [1, HELIO_COLORS[ii]]],
            showscale=False, sizeref=1.5, anchor='tail', showlegend=False))

        ray_src = hp + S * SUN_SCALE
        traces.append(go.Scatter3d(
            x=[ray_src[0], hp[0]], y=[ray_src[1], hp[1]], z=[ray_src[2], hp[2]],
            mode='lines', line=dict(color='gold', width=3, dash='dash'),
            name='Incident ray' if ii == 0 else None,
            showlegend=False))

    # ── Virtual aperture ────────────────────────────────────────────────────
    sun_vec = -np.array([sun.position.x, sun.position.y, sun.position.z])
    sun_vec /= np.linalg.norm(sun_vec)
    virt_centre, virt_aim, virt_W, virt_H = virtual_aperture(
        helio_positions, W, L, sun_vec, margin=2.0
    )
    virt_normal = virt_aim - virt_centre          # points back toward sun
    virt_normal /= np.linalg.norm(virt_normal)
    virt_pts = _panel_corners(virt_centre, virt_normal, virt_W, virt_H)
    all_pts.append(virt_pts)

    traces.append(go.Mesh3d(
        x=virt_pts[:,0], y=virt_pts[:,1], z=virt_pts[:,2],
        i=[0,0], j=[1,2], k=[2,3],
        color='mediumseagreen', opacity=0.25,
        name='Virtual aperture'))
    # outline
    outline = np.vstack([virt_pts, virt_pts[0]])
    traces.append(go.Scatter3d(
        x=outline[:,0], y=outline[:,1], z=outline[:,2],
        mode='lines', line=dict(color='mediumseagreen', width=3),
        showlegend=False))
    # ────────────────────────────────────────────────────────────────────────

    pts = _panel_corners(rec_pos, rec_norm, 5., 5.)
    all_pts.append(pts)
    traces.append(go.Mesh3d(
        x=pts[:,0], y=pts[:,1], z=pts[:,2],
        i=[0,0], j=[1,2], k=[2,3],
        color='firebrick', opacity=0.7, name='Receiver'))

    traces.append(go.Scatter3d(
        x=[rec_pos[0]]*2, y=[0., rec_pos[1]], z=[rec_pos[2]]*2,
        mode='lines', line=dict(color='dimgray', width=8), name='Tower'))

    scene_pts = np.vstack(all_pts)
    centroid  = scene_pts.mean(axis=0)
    radius    = np.max(np.linalg.norm(scene_pts - centroid, axis=1))
    sun_tip   = centroid + S * radius
    centroid += 0.5*(sun_tip - centroid)

    zshift = np.linspace(-centroid[2], centroid[2], 10)
    for ii, sh in enumerate(zshift):
        traces.append(go.Scatter3d(
            x=[sun_tip[0], centroid[0]], y=[sun_tip[1], centroid[1]],
            z=[sun_tip[2]+sh, centroid[2]+sh],
            mode='lines', line=dict(color='gold', width=4),
            legendgroup="sun_rays",
            name='Sun ray direction (plane waves)',
            showlegend=(ii==0)))
        traces.append(go.Cone(
            x=[centroid[0]], y=[centroid[1]], z=[centroid[2]+sh],
            u=[-S[0]], v=[-S[1]], w=[-S[2]],
            colorscale=[[0, 'gold'], [1, 'gold']],
            showscale=False, sizeref=radius * 0.02, anchor='tail'))

    fig = go.Figure(data=traces)
    fig.update_layout(
        scene=dict(xaxis_title='X West [m]', yaxis_title='Y Up [m]',
                   zaxis_title='Z North [m]', aspectmode='data',
                   camera=dict(eye=dict(x=3, y=0, z=0),
                               up=dict(x=0, y=1, z=0))),
        legend=dict(x=0.01, y=0.99))
    return fig

#### Plotly (interactive, browser)

In [ ]:
fig = build_geometry_fig()
fig.update_layout(title='Two-Heliostat Field — Geometry (Plotly)')
fig.show()

### Classify rays
Create a list detailing which element terminates each ray. Nans denote rays that are not terminated. 

In [ ]:
def classify_rays(raydata, nhat, virtual_stage=None):
    if virtual_stage is not None:
        raydata = raydata[raydata['stage'] != virtual_stage]
    
    elements = np.sort(raydata.element.abs().unique())
    num_rays = PT.raydata.number.max()   # full range, not unique count
    ray_end = np.nan * np.ones(num_rays)

    elements = elements[elements>0]
    for ii, e in enumerate(elements):
        absorbed_by_element = (raydata.element == -e)
        if any(absorbed_by_element):
            rda = raydata[absorbed_by_element]
            dp = rda.cos_x*nhat[ii][0] + rda.cos_y*nhat[ii][1] + rda.cos_z*nhat[ii][2]
            m_front = rda[dp < 0].number - 1
            m_back  = rda[dp > 0].number - 1
            ray_end[m_front] = e
            ray_end[m_back]  = -e

    return ray_end

In [ ]:
# Simulation needs to be inside of a __name__ guard to allow multi-threading
if __name__ == "__main__":

    # Run the configuration specified above
    PT.run(-1, False, 1)         #(seed, is point focus system?, number of threads)

    # Print a message after completion
    print("Num rays traced: {:d}".format(PT.raydata.index.size))
    
    # Generate a solatrace input file if desired
    PT.write_soltrace_input_file(od/'simpletest.stinput')

    # Create a 3D trace plot of the simulation (requires plotly library)
    # PT.plot_trace(show_sun_vector=False)

    nhat = helio_normals.copy()
    nhat.append(np.array(list(rec_norm)))
    ray_end = classify_rays(PT.raydata, nhat, virtual_stage=VIRTUAL_STAGE)

    PT.plot_flux(ela)

### Geometry + selected rays

Classifies rays from `PT.raydata` into three categories and overlays a sample of each on the geometry plot.

In [ ]:
import pandas as pd
view_distance = 1.0
N_THIN = 5000  # max scatter points per (element × outcome) — tunable

df = PT.raydata.copy()

VIRTUAL_STAGE = 1
df = df[df['stage'] != VIRTUAL_STAGE].copy()

# ray_end is 0-indexed (ray number − 1); align with 1-indexed ray numbers in raydata
ray_end_s = pd.Series(ray_end)
df['ray_end'] = (df['number'] - 1).map(ray_end_s)

rec_el = int(df['element'].abs().max())

def classify(val):
    if pd.isna(val):                  return 'miss'
    elif abs(int(val)) == rec_el:     return 'hit_receiver'
    elif val<0:                       return 'blocked'
    else:                             return 'absorbed'

df['outcome'] = df['ray_end'].apply(classify)

STYLES = {
    'blocked':          dict(color='crimson', name='Blocked'),
    'shaded':           dict(color='blue', name='Blocked'),
    'absorbed':         dict(color='crimson',   name='Lost'),
    'hit_receiver':     dict(color='limegreen', name='Hits receiver'),
}

fig = build_geometry_fig()
shown = set()

# Heliostat front-face hits
for el_id in range(1, rec_el):
    hits = df[df['element'] == el_id]
    for outcome, style in STYLES.items():
        sub = hits[hits['outcome'] == outcome]
        if sub.empty:
            continue
        if len(sub) > N_THIN:
            sub = sub.sample(N_THIN, random_state=42)
        sl = outcome not in shown
        shown.add(outcome)
        fig.add_trace(go.Scatter3d(
            x=sub['loc_x'], y=sub['loc_y'], z=sub['loc_z'],
            mode='markers',
            marker=dict(color=style['color'], size=2, opacity=0.5),
            name=style['name'],
            legendgroup=style['name'],
            showlegend=sl,
        ))

# Receiver hits (both faces)
rec_hits = df[df['element'].abs() == rec_el]
if not rec_hits.empty:
    sub = rec_hits if len(rec_hits) <= N_THIN else rec_hits.sample(N_THIN, random_state=42)
    style = STYLES['hit_receiver']
    fig.add_trace(go.Scatter3d(
        x=sub['loc_x'], y=sub['loc_y'], z=sub['loc_z'],
        mode='markers',
        marker=dict(color=style['color'], size=2, opacity=0.5),
        name=style['name'],
        legendgroup=style['name'],
        showlegend='hit_receiver' not in shown,
    ))

v = S*view_distance
fig.update_layout(title='Two-Heliostat Field — Hit Positions',
                  scene=dict(
                      camera=dict(
                          eye=dict(x=v[0], y=v[1], z=v[2]),
                          center=dict(x=centre[0], y=centre[1], z=centre[2]),
                          up=dict(x=0, y=1, z=0),  # Y=Up in SolTrace
                )
            ))
fig.show()


In [ ]:
from plotly.subplots import make_subplots

N_THIN_2D = 5000  # max scatter points per (element × outcome) — tunable
N_BINS     = 40  # heatmap bins per axis on receiver — tunable
LEGEND_SIZE = 20  # legend marker size (10× the scatter marker size of 2)

elements = st.elements   # all elements in the single stage, ordered by element id

def local_hits(df_hits, element):
    eu  = PT.util_calc_euler_angles(np.array(element.position.as_list()),
                                    np.array(element.aim.as_list()),
                                    element.zrot)
    tfm = PT.util_calc_transforms(eu)
    loc = df_hits[['loc_x', 'loc_y', 'loc_z']].to_numpy()
    res = PT.util_transform_to_local(loc, np.array([0., 0., 1.]),
                                     np.array(element.position.as_list()),
                                     tfm['rreftoloc'])
    return res['posloc'].T[0], res['posloc'].T[1]

n_el = len(elements)
titles = [f'Heliostat {i+1}' for i in range(n_el - 1)] + ['Receiver']
fig = make_subplots(rows=1, cols=n_el, subplot_titles=titles)

shown = set()
for col, (el_id, element) in enumerate(zip(range(1, n_el + 1), elements), start=1):
    W_ap, L_ap = element.aperture_params[0], element.aperture_params[1]
    is_receiver = (el_id == rec_el)

    el_hits = df[df['element'].abs() == el_id] if is_receiver else df[df['element'] == el_id]

    if is_receiver:
        xl, yl = local_hits(el_hits, element)
        fig.add_trace(go.Histogram2d(
            x=xl, y=yl,
            xbins=dict(start=-W_ap/2, end=W_ap/2, size=W_ap/N_BINS),
            ybins=dict(start=-L_ap/2, end=L_ap/2, size=L_ap/N_BINS),
            colorscale='Hot', showscale=True,
            colorbar=dict(title='Hits', thickness=12, len=0.6, x=1.02),
            name='Receiver flux', showlegend=False,
        ), row=1, col=col)
    else:
        for outcome, style in STYLES.items():
            sub = el_hits[el_hits['outcome'] == outcome]
            if sub.empty:
                continue
            if len(sub) > N_THIN_2D:
                sub = sub.sample(N_THIN_2D, random_state=42)
            xl, yl = local_hits(sub, element)

            # Legend entry: large marker, no data (shown once per category)
            if outcome not in shown:
                shown.add(outcome)
                fig.add_trace(go.Scatter(
                    x=[None], y=[None], mode='markers',
                    marker=dict(color=style['color'], size=LEGEND_SIZE),
                    name=style['name'], legendgroup=style['name'],
                    showlegend=True,
                ), row=1, col=1)

            # Data trace: small markers, no legend entry
            fig.add_trace(go.Scatter(
                x=xl, y=yl, mode='markers',
                marker=dict(color=style['color'], size=3, opacity=0.7),
                name=style['name'], legendgroup=style['name'],
                showlegend=False,
            ), row=1, col=col)

    x_ref = 'x' if col == 1 else f'x{col}'
    fig.update_xaxes(title_text='Local u [m]', range=[-W_ap/2, W_ap/2], row=1, col=col)
    fig.update_yaxes(title_text='Local v [m]', range=[-L_ap/2, L_ap/2],
                     scaleanchor=x_ref, scaleratio=1, row=1, col=col)

fig.update_layout(title='Hit positions in local element coordinates', height=450)
fig.show()
